# The Lazy Book Report

Your professor has assigned a book report on "The Red-Headed League" by Arthur Conan Doyle. 

You haven't read the book. And out of stubbornness, you won't.

But you *have* learned NLP. Let's use it to answer the professor's questions without reading.

## Setup

First, let's fetch the text from Project Gutenberg and prepare it for analysis.

In [1]:
# Fetch and prepare text - RUN THIS CELL FIRST
import os
import urllib.request
import re

os.makedirs("output", exist_ok=True)

url = 'https://www.gutenberg.org/files/1661/1661-0.txt'
req = urllib.request.Request(url, headers={'User-Agent': 'Python-urllib'})
with urllib.request.urlopen(req, timeout=30) as resp:
    text = resp.read().decode('utf-8')

# Strip Gutenberg boilerplate
text = text.split('*** START OF')[1].split('***')[1]
text = text.split('*** END OF')[0]

# Extract "The Red-Headed League" story (it's the second story in the collection)
matches = list(re.finditer(r'THE RED-HEADED LEAGUE', text, re.IGNORECASE))
story_start = matches[1].end()
story_text = text[story_start:]
story_end = re.search(r'\n\s*III\.\s*\n', story_text)
story_text = story_text[:story_end.start()] if story_end else story_text

# Split into 3 sections by word count
words = story_text.split()[:4000]
section_size = len(words) // 3
sections = [
    ' '.join(words[:section_size]),
    ' '.join(words[section_size:2*section_size]),
    ' '.join(words[2*section_size:])
]

print(f"Story loaded: {len(words)} words in {len(sections)} sections")
print(f"Section sizes: {[len(s.split()) for s in sections]}")

Story loaded: 4000 words in 3 sections
Section sizes: [1333, 1333, 1334]


## Professor's Questions

Your professor wants you to answer 5 questions about the story. Let's use NLP to find the answers.

---

## Question 1: Writing Style

> "This text is from the 1890s. What makes it different from modern writing?"

**NLP Method:** Use preprocessing to compute text statistics. Tokenize the text and calculate:
- Vocabulary richness (unique words / total words)
- Average sentence length
- Average word length

**Hint:** Formal, literary writing typically shows higher vocabulary richness and longer sentences than modern casual text.

In [ ]:
# Your code here: compute text statistics
# You'll need: import string, import re
import string 
import nltk
# - Tokenize: remove punctuation, lowercase
text = ' '.join(sections[0].split()) #remove extra whitespace 
Sentences = nltk.sent_tokenize(text)
text_clean = text.lower()
text_clean = text_clean.translate(str.maketrans('', '', string.punctuation))
Tokenize = nltk.word_tokenize(text_clean)
# - Sentences: split on sentence-ending punctuation
print(f"number of words: {len(Tokenize)}")
print(f"number of sentences: {len(Sentences)}")
# Calculate vocab_richness, avg_sentence_length, avg_word_length
vocab_richness = len(set(Tokenize))/len(Tokenize)
average_sentence_length = len(Tokenize)/len(Sentences)
average_word_length = sum(len(word) for word in Tokenize) / len(Tokenize)
print(f"vocab richness: {vocab_richness}")
print(f"sentence length: {average_sentence_length}")
print(f"word length: {average_word_length}")
#NOTE you have to sentence tokenize bedore you remove punctuation because without periods the sentence tokenizer cant work properly


number of words: 1412
number of sentences: 46
vocab richness: 0.40439093484419264
sentence length: 30.695652173913043
word length: 4.123937677053824


---

## Question 2: Main Characters

> "Who are the main characters in this story?"

**NLP Method:** Use Named Entity Recognition (NER) to extract PERSON entities.

**Hint:** Use spaCy's `en_core_web_sm` model. Process the text and filter entities where `ent.label_ == 'PERSON'`. Count how often each name appears.

In [17]:
# Your code here: extract PERSON entities using spaCy NER
# You'll need: import spacy, nlp = spacy.load("en_core_web_sm")
import spacy
nlp = spacy.load("en_core_web_sm")
text = sections[1]
doc = nlp(text)

characters = []
for ent in doc.ents:
    if ent.label_ == "PERSON":
        characters.append(ent.text)

print(f"Found {len(characters)} character mentions")
print(f"first ten: {characters[:10]}")

# When done, save your findings:
with open("output/characters.txt", "w") as f:
    for name in characters:
        f.write(f"{name}\n")



Found 14 character mentions
first ten: ['Wilson', 'Sherlock Holmes', 'Jabez Wilson', 'Sherlock Holmes', 'Vincent Spaulding', 'Holmes', 'Wilson', 'Wilson', 'Holmes', 'I. “‘Well']


---

## Question 3: Story Locations

> "Where does the story take place?"

**NLP Method:** Use Named Entity Recognition (NER) to extract location entities (GPE and LOC).

**Hint:** Filter entities where `ent.label_` is 'GPE' (geopolitical entity) or 'LOC' (location).

In [20]:
# Your code here: extract GPE and LOC entities using spaCy NER
text = sections[1]
doc = nlp(text)

locations = []
for ent in doc.ents:
    if ent.label_ == "LOC":
        locations.append(ent.text)
geopolitical = []
for ent in doc.ents:
    if ent.label_ == "GPE":
        geopolitical.append(ent.text)

print(f"Found {len(locations)} location mentions")
print(f"Found {len(geopolitical)} geopolitical mentions")

print(f"first ten locations: {locations[:10]}")
print(f"first ten geopolitical: {geopolitical[:10]}")

# When done, save your findings:
with open("output/locations.txt", "w") as f:
    for place in locations:
        f.write(f"{place}\n")



Found 1 location mentions
Found 1 geopolitical mentions
first ten locations: ['Londoners']
first ten geopolitical: ['London']


---

## Question 4: Wilson's Business

> "What is Wilson's business?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's business.

**Hint:** Create a TF-IDF vectorizer, fit it on the 3 sections, then transform your query using the same vectorizer (`.transform()`, not `.fit_transform()` - you want to use the vocabulary learned from the sections). Find which section has the highest cosine similarity and read it to find the answer.

In [ ]:
# Your code here: use TF-IDF similarity to find the relevant section
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


vectorizer = TfidfVectorizer()
section_vectors = vectorizer.fit_transform(sections)

query = "What is Wilson's business?"

# transform the query use  .transform, not .fit_transform)
query_vector = vectorizer.transform([query])

#similarity between query and each section
similarities = cosine_similarity(query_vector, section_vectors)

# which section has the highest similarity
best_section_idx = similarities.argmax()
best_similarity = similarities[0][best_section_idx]

print(f"Most relevant section: Section {best_section_idx}")
print(f"Similarity score: {best_similarity:.3f}")
print("\n--- Section content (first 500 characters) ---")
print(sections[best_section_idx][:500])

# When done, save your findings:
with open("output/business.txt", "w") as f:
    f.write("Wilson's business is: Wilson's business is: He is a pawnbroker with a small shop at Coburg Square, near the City.")
#for word, idf in zip(vectorizer.get_feature_names_out(), vectorizer.idf_):
#    print(f"{word}: IDF ={idf:.2f}")

# patient: IDF = 1.00   ← appears in all docs, lowest IDF
# denies: IDF = 1.69    ← appears in only 1 doc, high IDF
# When done, save your findings:



Most relevant section: Section 1
Similarity score: 0.147

--- Section content (first 500 characters) ---
your fortunes. You will first make a note, Doctor, of the paper and the date.” “It is _The Morning Chronicle_ of April 27, 1890. Just two months ago.” “Very good. Now, Mr. Wilson?” “Well, it is just as I have been telling you, Mr. Sherlock Holmes,” said Jabez Wilson, mopping his forehead; “I have a small pawnbroker’s business at Coburg Square, near the City. It’s not a very large affair, and of late years it has not done more than just give me a living. I used to be able to keep two assistants, 
1890: IDF =1.29
27: IDF =1.69
_employé_: IDF =1.69
_encyclopædia: IDF =1.69
_omne: IDF =1.69
_the: IDF =1.69
abbots: IDF =1.69
able: IDF =1.00
about: IDF =1.00
above: IDF =1.69
abruptly: IDF =1.69
account: IDF =1.69
acknowledges: IDF =1.69
addition: IDF =1.69
address: IDF =1.69
admirably: IDF =1.69
admit: IDF =1.69
adventures: IDF =1.69
advertisement: IDF =1.29
affair: IDF =1.29
afraid: IDF 

---

## Question 5: Wilson's Work Routine

> "What is Wilson's daily work routine for the League?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's work routine.

**Hint:** Similar to Question 4 - use TF-IDF to find the section that best matches your query about work routine. The answer includes what Wilson had to do and what eventually happened.

In [ ]:
# Your code here: use TF-IDF similarity to find the relevant section

vectorizer = TfidfVectorizer()
section_vectors = vectorizer.fit_transform(sections)

query = "What is Wilson's daily work routine for the League?"

# transform the query use  .transform, not .fit_transform)
query_vector = vectorizer.transform([query])

#similarity between query and each section
similarities = cosine_similarity(query_vector, section_vectors)

# which section has the highest similarity
best_section_idx = similarities.argmax()
best_similarity = similarities[0][best_section_idx]

print(f"Most relevant section: Section {best_section_idx}")
print(f"Similarity score: {best_similarity:.3f}")
print("\n--- Section content ( 500 characters) ---")
print(sections[best_section_idx])
for word, idf in zip(vectorizer.get_feature_names_out(), vectorizer.idf_):
    print(f"{word}: IDF ={idf:.2f}")




# When done, save your findings:
with open("output/routine.txt", "w") as f:
    f.write("Wilson's work routine: Ten to two mostly done in the evening especially thursday and friday. 4 pounds a day. Its only a few hours and you cant leave when you are there you have to copy the encyclopedia britannica with your own ink pen and blotting paper\n")
    f.write("What happened: After 8 weeks the league dissolved \n")



Most relevant section: Section 2
Similarity score: 0.285

--- Section content ( 500 characters) ---
any of the others, and he closed the door as we entered, so that he might have a private word with us. “‘This is Mr. Jabez Wilson,’ said my assistant, ‘and he is willing to fill a vacancy in the League.’ “‘And he is admirably suited for it,’ the other answered. ‘He has every requirement. I cannot recall when I have seen anything so fine.’ He took a step backward, cocked his head on one side, and gazed at my hair until I felt quite bashful. Then suddenly he plunged forward, wrung my hand, and congratulated me warmly on my success. “‘It would be injustice to hesitate,’ said he. ‘You will, however, I am sure, excuse me for taking an obvious precaution.’ With that he seized my hair in both his hands, and tugged until I yelled with the pain. ‘There is water in your eyes,’ said he as he released me. ‘I perceive that all is as it should be. But we have to be careful, for we have twice been dece